This model uses the single-cell measurement data, NOT distance matrix \
here here every row (input unit) is a single cell

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, LeaveOneOut
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from pathlib import Path
from sklearn.utils import shuffle

from sklearn.svm import SVC, LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.inspection import DecisionBoundaryDisplay

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import math
seed = 114514
ecad = 'E-CadGFP'
gap43 = 'gap43-mCherry'

In [ ]:
# read csv files
dataset_dir = Path("./raw_dataset")
frames = []
i = 0

#NOTE: because of LOO validation, do not split train and test set
for csv_path in dataset_dir.rglob('*.csv'):
    df = pd.read_csv(csv_path)
    path = str(csv_path)
    
    #keep track of embryos
    df['EM'] = i  #note: do NOT include 'EM' in cell-wise RF 
    i += 1

    #strain
    if gap43 in path:
        df['strain'] = gap43
    elif ecad in path:
        df['strain'] = ecad
    #end if
    
    #control or mutant
    if 'control' in path:
        df['label'] = 0
    elif 'mutant' in path:
        df['label'] = 1
    #end if

    frames.append(df)
#end for

df = pd.concat(frames).reset_index(drop = True)
df = df.drop(columns = [' ', 'X', 'Y', 'FeretX', 'FeretY'])

In [ ]:
# weighting
# AR (major axis/ minor axis) is used instead of circularity
# because even regular polygons don't have a circularity of 1 ?
df = df[df['AR'] > 1.5 ]

In [ ]:
print('All Data')
display(df.iloc[[100*i for i in range(20)], :])

Inspect Correlation to Remove Redundant Features

In [ ]:
#create correlation matrix
cols = df.columns.drop(['EM', 'strain', 'label'])  #feature-only columns

corr_matrix = df[cols].corr(method = 'pearson')
display(corr_matrix)

#you'll notice that the correlation between Angle and FeretAngle is unexpectedly low
#this is because angles are periodic
#they need to be shifted to the correct value for the true correlation to be revealed

In [ ]:
# make a correlation scatter plot
def corr_plot(feature_1: str, feature_2: str, df = df, hue = 'label'):
    fig, ax = plt.subplots(1, 2, figsize = (12, 5), sharex = True, sharey = True)
    sns.scatterplot(data = df[df['strain'] == ecad], x = feature_1, y = feature_2, hue = hue, ax = ax[0])
    sns.scatterplot(data = df[df['strain'] == gap43], x = feature_1, y = feature_2, hue = hue, ax = ax[1])
    ax[0].set_title(f'{ecad}: {feature_1} vs {feature_2}')
    ax[1].set_title(f'{gap43}: {feature_1} vs {feature_2}')
    plt.show()
#end def

In [ ]:
corr_plot('Area', 'Perim.')

In [ ]:
corr_plot('Angle', 'FeretAngle')

In [ ]:
# shift angle data to appropriate positions
idx = list(df[((df['Angle'] > 150) & (df['FeretAngle'] < 30)) | ((df['FeretAngle'] > 150) & (df['Angle'] < 30))].index)

df_copy = df.copy()
for i in idx:
    if df_copy.loc[i, 'Angle'] > 150:
        df_copy.loc[i, 'Angle'] = df_copy.loc[i, 'Angle'] - 180
    else:
        df_copy.loc[i, 'FeretAngle'] = df_copy.loc[i, 'FeretAngle'] - 180
    #end if
#end for

In [ ]:
#the corr plot after shifting
corr_plot('Angle', 'FeretAngle', df = df_copy)

In [ ]:
#check for correlation between features
corr_matrix = df_copy[cols].corr(method = 'pearson')
display(corr_matrix)

#now the true correlation is ~98%

In [ ]:
#remove correlated features in the correlation matrix
thres = 0.75
to_remove = set()
for i in range(len(cols)-1):
    for j in range(i+1, len(cols)):
        if abs(corr_matrix.iloc[i, j]) > 0.75:
            to_remove.add(cols[j])
        #end if
    #end for
#end for
cols_to_remove = list(to_remove)
reduced_corr_matrix = corr_matrix.drop(cols_to_remove, axis = 0, inplace = False)
reduced_corr_matrix = reduced_corr_matrix.drop(cols_to_remove, axis = 1, inplace = False)
reduced_corr_matrix

In [ ]:
#remove redundant features in the dataframe and cols
reduced_cols = reduced_corr_matrix.columns
reduced_df = df.drop(columns = cols_to_remove, inplace = False)
reduced_df

Plot Feature Distribution

In [ ]:
#compare control and mutant of the same strain
def compare(strain: str, feature: str, df = df, n_bins = 16):
    def plot_EMs(lbl: int):
        sub_df = df[(df['strain'] == strain) & (df['label'] == lbl)]
        EMs = list(set(sub_df['EM']))
        n = len(EMs)

        n_col = 4
        n_row = math.ceil(n/n_col)
        fig, ax = plt.subplots(nrows = n_row, ncols = n_col, figsize = (3*n_col, 3*n_row), sharey = True, sharex = True)
        ax = ax.flatten()

        #bins will the SAME since we're referring to the same feature column
        bins = np.linspace(df[feature].min(), df[feature].max(), n_bins + 1)
        
        for i, EM in enumerate(EMs):
            sns.histplot(data = sub_df[sub_df['EM'] == EM], x=feature, bins = bins, ax = ax[i], kde=True, stat = 'percent')

            phenotype = 'Control' if lbl == 0 else 'Mutant'
            ax[i].set_title(f'{strain} {phenotype} {feature} EM-{EM}', fontsize = 8)
        #end for
        
        plt.tight_layout()
        plt.show()         
    #end def

    plot_EMs(0)
    plot_EMs(1)
    
#end def

In [ ]:
# for making plots for the report
'''
sub_df = df[df['strain'] == ecad]
feature = 'Angle'
strain = ecad
bins = np.linspace(df[feature].min(), df[feature].max(), 17)
fig, ax = plt.subplots(1, 2, figsize = (6, 3), sharey = True, sharex = True)
sns.histplot(data = sub_df[sub_df['EM'] == 4], x=feature, bins = bins, ax = ax[0], kde=True, stat = 'percent')
sns.histplot(data = sub_df[sub_df['EM'] == 8], x=feature, bins = bins, ax = ax[1], kde=True, stat = 'percent')
ax[0].set_title(f'{strain} Control {feature} EM-{4}', fontsize = 8)
ax[1].set_title(f'{strain} Mutant {feature} EM-{8}', fontsize = 8)
plt.show() 
'''

In [ ]:
# a particularly interesting observation is that gap43 control and mutants have distinct angle distribution
# but this is not the case for E-Cad
compare(gap43, 'Angle')

In [ ]:
compare(ecad, 'Angle')

Convert Embryo data into bins

In [ ]:
# run to see how pd.cut works
'''
d = pd.DataFrame({'A':[5,4,3,2,1,1]})
a, bins = pd.cut(x = d['A'], 
           bins = 5,
           labels = [('A' + '_' + str(i)) for i in range(5)], 
           ordered = True,
           retbins = True)
print(a, end = '\n\n')
print(bins)
'''

In [ ]:
# run to see how pd.pivot_table works
'''
nan = np.nan

data = {
    'tissue_id': ['heart', 'heart', 'heart', 'lung', 'lung', 'liver'],
    'cell_type': ['cardiomyocyte', 'fibroblast', nan, 'macrophage', 'macrophage', 'hepatocyte']
}
d = pd.DataFrame(data)
display(d)

pivot_d = pd.pivot_table(data = d,
                         index='tissue_id', 
                         columns='cell_type', 
                         aggfunc='size', 
                         fill_value=0) # Replaces empty combinations with 0 instead of NaN
display(pivot_d)
'''

In [ ]:
#stores bin boundaries for each feature
bin_dict = {}

In [ ]:
#use for reduced_df or df
def make_bins(df, num_bins = 16):
    temp_df = df.copy()  #use temp_df so df does NOT get modified
    
    profile_frames = []
    cols = temp_df.columns.drop(['EM', 'strain', 'label'])

    for col in cols:
        if len(bin_dict) < len(reduced_cols):  # if bins were not specified, then specify bins 
            temp_df[col + '_bins'], bins = pd.cut(x = temp_df[col], 
                                                  bins = num_bins,
                                                  labels = [(col + '_bin_' + str(i)) for i in range(num_bins)],
                                                  retbins = True)
            bin_dict[col] = bins    
        else:  # if bins were specified, then use existing bins. this ensures that we use uniform bins
            temp_df[col + '_bins'] = pd.cut(x = temp_df[col], 
                                            bins = bin_dict[col],
                                            labels = [(col + '_bin_' + str(i)) for i in range(num_bins)])
        #end if
    
        profile = temp_df.pivot_table(index = ['EM', 'strain', 'label'], 
                                      columns = col + '_bins', 
                                      aggfunc = 'size', 
                                      fill_value = 0,
                                      observed = False)

        #remove redundant all-0 rows
        profile = profile[profile.sum(axis = 1) > 0].astype('float64')

        #convert to percentages
        for i in range(len(profile)):
            profile.iloc[i, :] = profile.iloc[i, :] / profile.iloc[i, :].sum() * 100
        #end for
    
        profile_frames.append(profile)
    #end for
    
    binned_df = pd.concat(profile_frames, axis = 1)
    return binned_df
#end def

In [ ]:
binned_reduced_df = make_bins(reduced_df)
binned_reduced_df.head()

In [ ]:
# train on gap43
X_train = binned_reduced_df.xs(gap43, level = 1)
y_train = [lbl for (em, lbl) in X_train.index]

# test on ecad
X_test = binned_reduced_df.xs(ecad, level = 1)
y_test = [lbl for (em, lbl) in X_test.index]

display(X_test)

Train the Random Forest

In [ ]:
my_RF = RandomForestClassifier(n_estimators = 100, 
                               max_depth = 2,
                               max_features = 0.3,
                               min_samples_leaf = 2,
                               random_state = seed)#.fit(X_train, y_train)

In [ ]:
# use LeaveOneOut to justify the choice of model
score = cross_val_score(my_RF, X_train, y_train, cv = LeaveOneOut())
print(f'RF LeaveOneOut Result: {score}')
print(f'accuracy = {sum(score)/len(score) * 100:.2f}%')

In [ ]:
# train the model on gap43 and test on E-Cad 
my_RF.fit(X_train, y_train)
y_pred = list(my_RF.predict(X_test))
score = f1_score(y_test, y_pred, average = 'macro')

print('RF Testing:')
print(f'{y_test = }')
print(f'{y_pred = }')
print(f'F1 score = {score*100:.2f}%')  #very bad performance on E-Cad controls

In [ ]:
# Check feature importance: 
# imp for importances
imp_rf = pd.DataFrame(columns = X_train.columns, 
                      data = [my_RF.feature_importances_*100]) #to percentage

In [ ]:
imp_rf.drop(columns = [col for col in imp_rf.columns if imp_rf[col][0] < 5], inplace=True)
print('Feature Importances (>5%) :')
display(imp_rf)

# the model is capturing a lot of angle bins as important features

Reduce X_train and X_test to only the importance features

In [ ]:
X_train_reduced = X_train[imp_rf.columns]
X_test_reduced = X_test[imp_rf.columns]

# re-evaluate and re-test to see performance
# not very different
score= cross_val_score(my_RF, X_train_reduced, y_train, cv = LeaveOneOut())
print(f'RF LeaveOneOut Result: {score}')
print(f'accuracy = {sum(score)/len(score) * 100:.2f}%')
print()

my_RF.fit(X_train_reduced, y_train)
y_pred = list(my_RF.predict(X_test_reduced))
score = f1_score(y_test, y_pred, average = 'macro')
print('RF Testing:')
print(f'{y_test = }')
print(f'{y_pred = }')
print(f'F1 score = {score*100:.2f}%')  #very bad performance on E-Cad controls

Train the Support Vector Machine

In [ ]:
clf = LinearSVC(penalty = 'l2', 
                C = 1, 
                dual = 'auto',
                random_state=0)

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns = X_train.columns) #now X is transformed, each col mean = 0, std = 1
                   
score = cross_val_score(clf, X_train_scaled, y_train, cv = LeaveOneOut())
print(f'SVM LeaveOneOut Result: {score}')
print(f'accuracy = {sum(score)/len(score) * 100:.2f}%')

In [ ]:
# train the model on gap43 and test on E-Cad 
clf.fit(X_train_scaled, y_train)

X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns = X_test.columns)
y_pred = list(clf.predict(X_test_scaled))
score = f1_score(y_test, y_pred, average = 'macro')

print('SVM Testing:')
print(f'{y_test = }')
print(f'{y_pred = }')
print(f'F1 score = {score*100:.2f}%')
#misclassifying most E-Cad controls wrong, consistent with the abnormal angle distribution

In [ ]:
# check feature importance
# relative importances is given by the normal vector of the decision hyperplane

# normalise the normal vector
normal_vec = clf.coef_[0]
norm = np.sqrt(sum([i**2 for i in normal_vec]))
normal_vec = normal_vec / norm
importance = [100*x**2 for x in normal_vec]

imp_clf = pd.DataFrame(data = [importance], columns = X_train.columns)
imp_clf.drop(columns = [col for col in imp_clf.columns if imp_clf[col][0] < 5], inplace=True)
print('Feature Importances (>5%) :')
display(imp_clf)

#importance of angles nicely captured

Reduce X_train and X_test to only the importance features

In [ ]:
X_train_scaled_reduced = X_train_scaled[imp_clf.columns]
X_test_scaled_reduced = X_test_scaled[imp_clf.columns]


score = cross_val_score(clf, X_train_scaled, y_train, cv = LeaveOneOut())
print(f'SVM LeaveOneOut Result: {score}')
print(f'accuracy = {sum(score)/len(score) * 100:.2f}%')
print()


# train the model on gap43 and test on E-Cad 
clf.fit(X_train_scaled_reduced, y_train)

y_pred = list(clf.predict(X_test_scaled_reduced))
score = f1_score(y_test, y_pred, average = 'macro')

print('SVM Testing:')
print(f'{y_test = }')
print(f'{y_pred = }')
print(f'F1 score = {score*100:.2f}%')
#misclassifying most E-Cad controls wrong, consistent with the abnormal angle distribution

Project the reduced data points and classification boundary onto a 2D plane

In [ ]:
clf.predict(X_train_scaled_reduced) == y_train

Projection onto 2D Feature Space

In [ ]:
# 1. get the top 2 most important features
l = len(importance)
(idx_1, _), (idx_2, _) = sorted(enumerate(imp_clf), key=lambda x: x[1])[-2:]

# 2. project all vectors onto the 2D feature component space
X_2d = np.zeros((len(X_train_scaled), 2))
X_2d[:, 0] = X_train_scaled_reduced.iloc[:, idx_1]
X_2d[:, 1] = X_train_scaled_reduced.iloc[:, idx_2]

# 3. Create a 2D grid in the feature component space
x_min, x_max = X_train_scaled_reduced.iloc[:, idx_1].min() - 1,  X_train_scaled_reduced.iloc[:, idx_1].max() + 1
y_min, y_max = X_train_scaled_reduced.iloc[:, idx_2].min() - 1,  X_train_scaled_reduced.iloc[:, idx_2].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

# 4. Prepare 2D grid points
grid_2d = np.c_[xx.ravel(), yy.ravel()]

# 5. INVERSE TRANSFORM: Map 2D grid points back to high-D vectors
grid_high_d = np.zeros((len(grid_2d), len(imp_clf.columns)))
grid_high_d[:, idx_1] = grid_2d[:, 0]
grid_high_d[:, idx_2] = grid_2d[:, 1]

# 6. Predict using the REAL model
Z = clf.decision_function(grid_high_d).reshape(xx.shape)

# 7. Plot the true projected boundary
# datapoints
plt.scatter(X_2d[:, 0], 
            X_2d[:, 1], 
            c = y_train, 
            edgecolors = 'k')

# decision plane
plt.contour(xx, yy, Z, 
            levels = [-1, 0, 1],                      # Draw lines at -1, 0, +1
            colors = ['black', 'black', 'black'],
            linestyles = ['--', '-', '--'],           # Dashed margins, solid decision line
            linewidths = [1.5, 2.5, 1.5])

# background colour
plt.contourf(xx, yy, Z, cmap='coolwarm', alpha=0.2)

plt.xlabel(X_train_scaled_reduced.columns[idx_1])
plt.ylabel(X_train_scaled_reduced.columns[idx_2])
plt.title('Projection of SVM Decision Hyperplane onto 2D Feature Space')
plt.show()


Projection onto 2D PCA Space

In [ ]:
# 1. Fit PCA to learn the 2D projection transformation
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_train_scaled_reduced)  #mean 0

# 2. Train the SVM on high-dimensional data
clf.fit(X_train_scaled_reduced, y_train) 

# 3. Create a 2D grid in the PCA component space
x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

# 4. Prepare 2D grid points
grid_2d = np.c_[xx.ravel(), yy.ravel()]

# 5. INVERSE TRANSFORM: Map 2D grid points back to high-D vectors
# eg: a 2D dot of (1,2) will be mapped to 1*u1 + 2*u2, where u1 and u2 are the first 2 principal components
# the collection of 2D dots will be mapped to a 2D hyperplane in high-D space, and the decision plane will cut through it
grid_high_d = pca.inverse_transform(grid_2d)

# 6. Predict using the REAL model
Z = clf.decision_function(grid_high_d).reshape(xx.shape)

# 7. Plot the true projected boundary
# datapoints
plt.scatter(X_2d[:, 0], 
            X_2d[:, 1], 
            c = y_train, 
            edgecolors = 'k')

# decision plane
plt.contour(xx, yy, Z, 
            levels = [-1, 0, 1],                      # Draw lines at -1, 0, +1
            colors = ['black', 'black', 'black'],
            linestyles = ['--', '-', '--'],           # Dashed margins, solid decision line
            linewidths = [1.5, 2.5, 1.5])

# background colour
plt.contourf(xx, yy, Z, cmap='coolwarm', alpha=0.2)

plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.title('Projection of SVM Decision Hyperplane onto PCA Space')
plt.show()

Testing with Fixed Embryos (stage 7) \
can use stage 8, similar results

In [ ]:
strain = 'yw'
root_dir = Path(r"C:\Users\ethan\OneDrive - University of Cambridge\fixed_EM\processed")

# read csv files of corresponding strain
def get_df_test(strain: str):
    dataset_dir = root_dir / strain
    frames = []
    i = 0
    
    for csv_path in dataset_dir.rglob('measurement.csv'):
        temp = pd.read_csv(csv_path)
        path = str(csv_path)
    
        #only want stage 8 embryos
        if 's7' in path:
            #keep track of embryos
            temp['EM'] = i  #note: do NOT include 'EM' in cell-wise RF 
            i += 1
    
            temp['strain'] = strain

            if strain == 'yw' or strain == 'del7':
                temp['label'] = 0
            else:
                temp['label'] = 1
            #end if
    
            frames.append(temp)
        #end if
    #end for
    df_test = pd.concat(frames).reset_index(drop = True)
    df_test = df_test[df_test['AR'] > 1.5 ]
    df_test = df_test[reduced_cols].join(df_test[['EM', 'strain', 'label']])

    return df_test
#end def

In [ ]:
def test(strain: str):
    df_test = get_df_test(strain)

    if bin_dict != {}:
        X_test = make_bins(df_test)
    else:
        print('Model is un-trained')
        return
    #end if
    
    X_test = X_test.xs(strain, level = 1)
    y_test = [lbl for (em, lbl) in X_test.index]

    # RF testing:
    X_test_rf = X_test[imp_rf.columns]
    y_pred = list(my_RF.predict(X_test_rf))
    
    print('RF Testing:')
    print(f'{y_test = }')
    print(f'{y_pred = }')
    print(f'score = {f1_score(y_test, y_pred, average = 'macro')}')

    print('---------------------------')

    # SVM testing: 
    X_test_svm = pd.DataFrame(scaler.transform(X_test), columns = X_test.columns)
    X_test_svm = X_test_svm[imp_clf.columns]
    y_pred = list(clf.predict(X_test_svm))
    
    print('SVM Testing')
    print(f'{y_test = }')
    print(f'{y_pred = }')
    print(f'score = {f1_score(y_test, y_pred, average = 'macro')}')
#end def

yw

In [ ]:
test('yw')

sdk5054

In [ ]:
test('sdk5054')

del15 (no s8 embryos)

In [ ]:
test('del15')

del7 

In [ ]:
test('del7')

Try clustering

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
def cluster(n_clusters=5):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df_all['cluster'] = kmeans.fit_predict(X_scaled)
    
    # Evaluate clustering quality (Silhouette Score: -1 to +1)
    score = silhouette_score(X, df_all['cluster'])
    return score # , X_scaled
#end def

In [ ]:
# s8 no del15

In [ ]:
df_yw = make_bins(get_df_test('yw'))
df_sdk5054 = make_bins(get_df_test('sdk5054'))
df_del7 = make_bins(get_df_test('del7'))
df_del15 = make_bins(get_df_test('del15'))

df_all = pd.concat([binned_reduced_df, df_yw, df_sdk5054, df_del7, df_del15], axis = 0)

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(df_all), columns = X_train.columns)
X = X[imp_rf.columns]

In [ ]:
# store y values for elbow plot
y = []
for i in range(2, 10):
    y.append(cluster(i))
#end for

In [ ]:
x = [i for i in range(2,10)]
plt.plot(x, y)
plt.title('Elbow Plot')
plt.show()

# 5 appears to be the best natural cluster number. we also have 5 different genotypic backgrounds

In [ ]:
cluster(5)

In [ ]:
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)  #mean 0

Project the clusters onto 2D PCA space

In [ ]:
# grouped by natural clusters and phenotypes (control, mutant)

fig, ax = plt.subplots(1, 2, figsize = (12, 4), sharex = True, sharey = True)
sns.scatterplot(
    x=X_2d[:, 0], 
    y=X_2d[:, 1], 
    hue=df_all['cluster'], 
    palette='tab10',
    edgecolor='k',
    s=50,
    ax = ax[0]
)

ax[0].set_xlabel('PC 1')
ax[0].set_ylabel('PC 2')
ax[0].set_title('Projection of Clustering onto PCA Space')
ax[0].legend(title='unsupervised clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

sns.scatterplot(
    x=X_2d[:, 0], 
    y=X_2d[:, 1], 
    hue=df_all.index.get_level_values('label'), 
    palette='tab10',
    edgecolor='k',
    s=50,
    ax = ax[1]
)

ax[1].set_xlabel('PC 1')
ax[1].set_ylabel('PC 2')
ax[1].set_title('Control vs Mutants')
ax[1].legend(title='phenotype', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# assign a genotype label to every embryo
strain_num = []
for (em, strain, lbl) in df_all.index:
    if (strain == gap43 and lbl == 0) or (strain == ecad and lbl == 0): # wt
        num = 0
    elif strain == 'del7': # del7
        num = 1
    elif strain == 'yw': # yw
        num = 2
    elif (strain == gap43 and lbl == 1) or (strain == 'sdk5054'): # sdk5054
        num = 3
    elif (strain == ecad and lbl == 1) or (strain == 'del15'): # del15
        num = 4
    #end if

    strain_num.append(num)
#end for

In [ ]:
# group by natural clusters and genotypes

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# 1. Make sure PCA learns 3 components for 3D plotting
pca = PCA(n_components=3)
X_3d = pca.fit_transform(X_scaled)

# 2. Convert clusters to numeric values (handles strings & index mismatches)
if df_all['cluster'].dtype == 'object' or isinstance(df_all['cluster'].iloc[0], str):
    cluster_numeric = df_all['cluster'].astype('category').cat.codes.values
else:
    cluster_numeric = df_all['cluster'].values

# 3. Create 3D subplots
fig, ax = plt.subplots(
    1, 2, 
    figsize=(14, 8), 
    subplot_kw={'projection': '3d'}
)

# 4. Plot on Subplot 1
scatter0 = ax[0].scatter(
    X_3d[:, 0], 
    X_3d[:, 1], 
    X_3d[:, 2], 
    c=cluster_numeric, 
    cmap='tab10', 
    edgecolor='k', 
    s=60,            # Marker size
    alpha=0.8
)

ax[0].set_xlabel('PC 1')
ax[0].set_ylabel('PC 2')
ax[0].set_zlabel('PC 3')
ax[0].set_title('Projection of Clustering onto PCA Space')
ax[0].legend(*scatter0.legend_elements(), title="unsupervised cluster", bbox_to_anchor=(1.15, 1))

# 5. Plot on Subplot 2
scatter1 = ax[1].scatter(
    X_3d[:, 0], 
    X_3d[:, 1], 
    X_3d[:, 2], 
    c=strain_num, 
    cmap='coolwarm', 
    edgecolor='k', 
    s=60,
    alpha=0.8
)

ax[1].set_xlabel('PC 1')
ax[1].set_ylabel('PC 2')
ax[1].set_zlabel('PC 3')
ax[1].set_title('Coloured by Genotype')
ax[1].legend(*scatter1.legend_elements(), title="genotype", bbox_to_anchor=(1.15, 1))

plt.tight_layout()
plt.show()